# 🏗️ Session 05 — Structured Outputs & Reliable Tool Pipelines

## Making Agents Production-Ready: Structured, Validated, Safe

| Detail | Info |
|---|---|
| **Session** | 05 (Module 3) |
| **Duration** | ~2 hours |
| **Prerequisite** | Session 04 (AI Agents & Tool Usage) |
| **Environment** | Local (VS Code / Jupyter) or Google Colab |
| **Libraries** | `openai`, `python-dotenv`, `json` |
| **API Key** | OpenAI key from `.env` file |

### What We've Covered So Far

| Session | Topic | Key Takeaway |
|---|---|---|
| 01 | LLM Core Concepts | Tokens, context windows, temperature |
| 02 | Prompt Engineering | How to talk to the LLM effectively |
| 03 | Embeddings & Search | Text → meaning vectors |
| 04 | AI Agents & Tools | LLM + Tools + Loop = Agent |
| **05** | **Structured Outputs** | **Making agent responses reliable & predictable** |

### The Problem We're Solving Today

In Session 04, our agent worked great — but the LLM's responses were **free text**. Ask it the same question twice, you might get:

```
Attempt 1: "The average salary is ₹65,000"
Attempt 2: "Based on my analysis, the mean salary comes to approximately 65000 rupees"
Attempt 3: "65000"
```

All correct, but if your **code** needs to extract the number — which format do you parse? This breaks pipelines.

> 💡 **Today's Goal**: Force the LLM to respond in an EXACT structure, validate it, and build tools that are safe to re-run.

---
## Part 1: The Problem with Free Text

Let's see the problem live. We'll ask the LLM the same question 3 times and look at the format:

In [ ]:
!pip3 install openai python-dotenv -q

In [ ]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv('openaiapikey'))

print("✅ Setup complete!")

In [ ]:
# Ask the same question 3 times — notice the format changes each time

prompt = ''' Summarize this customer request and identify what they need: 
            'Hi, I was looking at your website and I think I want to get 
            some of those Samsung phones for my team -
            probably around 3 of them, and I saw they
            were listed at around 25k each, 
            but I'm not sure if that includes GST or not.
            Can you also check if there's a bulk discount?' '''

print("Asking the SAME question 3 times (with temperature=1 for variety):")
print("=" * 60)

for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0
    )
    print(f"\nAttempt {i+1}:")
    print(response.choices[0].message.content)
    print("-" * 40)

See the problem? Each time the format is different:
- Sometimes bullet points, sometimes a paragraph
- Sometimes with ₹ symbol, sometimes without
- Sometimes extra explanation, sometimes just values

What if the question is like this - I want to buy 3 Samsung Galaxy phones at Rs 25000 each

If your **downstream code** expects `{"product": "...", "price": 25000, "qty": 3}` — you're in trouble.

```
┌────────────────────────────────────────────────┐
│  WITHOUT Structured Outputs                    │
│                                                │
│  LLM → "The price is Rs 25000"    ← free text │
│  LLM → "Price: ₹25,000"          ← different │
│  LLM → "25000 rupees per unit"    ← different │
│                                                │
│  Your code: HOW DO I PARSE THIS?! 😱          │
└────────────────────────────────────────────────┘

┌────────────────────────────────────────────────┐
│  WITH Structured Outputs                       │
│                                                │
│  LLM → {"product": "Samsung Galaxy",           │
│          "price": 25000,                       │
│          "quantity": 3}                        │
│                                                │
│  EVERY. SINGLE. TIME. ✅                       │
└────────────────────────────────────────────────┘
```

---
## Part 2: Structured Outputs — Force the LLM Into a Shape

OpenAI has a feature called `response_format` that **guarantees** the LLM's output is valid JSON matching a schema you define.

### How It Works

```
YOU define:    "I want {product: string, price: number, quantity: number}"
LLM outputs:   {"product": "Samsung Galaxy", "price": 25000, "quantity": 3}
               ↑ GUARANTEED to match your schema. Always.
```

It's like giving someone a form to fill instead of asking them to write a free essay. The form has boxes — they MUST put the answer in the right box.

In [6]:
# ──────────────────────────────────────
# Method 1: Simple JSON mode (json_object)
# ──────────────────────────────────────
# This just forces the output to be VALID JSON (any shape)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You extract product info. Always respond in JSON with keys: product, price, quantity."},
        {"role": "user", "content": "I want to buy 3 Samsung Galaxy phones at Rs 25000 each"}
    ],
    response_format={"type": "json_object"}  # ← Forces valid JSON output
)

raw = response.choices[0].message.content
print("Raw output (string):")
print(raw)
print()

# Parse it into a Python dict
data = json.loads(raw)
print("Parsed (dict):")
print(f"  Product:  {data['product']}")
print(f"  Price:    ₹{data['price']}")
print(f"  Quantity: {data['quantity']}")
print(f"  Total:    ₹{data['price'] * data['quantity']}")

Raw output (string):
{
  "product": "Samsung Galaxy phone",
  "price": 25000,
  "quantity": 3
}

Parsed (dict):
  Product:  Samsung Galaxy phone
  Price:    ₹25000
  Quantity: 3
  Total:    ₹75000


> 💡 **Key Insight**: With `response_format={"type": "json_object"}`, the LLM is FORCED to output valid JSON. No more free text. But it doesn't guarantee the *specific keys* — the LLM could still choose different key names.

For full control over the exact shape, we use **JSON Schema** mode:

In [9]:
# ──────────────────────────────────────
# Method 2: Strict JSON Schema (full control)
# ──────────────────────────────────────
# This forces the output to match an EXACT schema

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract product information from the user's message."},
        {"role": "user", "content": "I want to buy 3 Samsung Galaxy phones at Rs 25000 each"}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "product_extraction",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "Name of the product"
                    },
                    "price_per_unit": {
                        "type": "number",
                        "description": "Price of one unit in INR"
                    },
                    "quantity": {
                        "type": "integer",
                        "description": "Number of units requested"
                    },
                    "total_amount": {
                        "type": "number",
                        "description": "Total cost (price × quantity)"
                    }
                },
                "required": ["product_name", "price_per_unit", "quantity", "total_amount"],
                "additionalProperties": False
            }
        }
    }
)

result = json.loads(response.choices[0].message.content)
print("Structured output (guaranteed shape):")
print(json.dumps(result, indent=2))

Structured output (guaranteed shape):
{
  "product_name": "Samsung Galaxy phone",
  "price_per_unit": 25000,
  "quantity": 3,
  "total_amount": 75000
}


In [ ]:
# Run it 3 times — the SHAPE is always the same!

texts = [
    "I want to buy 3 Samsung Galaxy phones at Rs 25000 each",
    "Please order 10 notebooks for 50 rupees per piece",
    "Get me 2 MacBook Pro laptops, each costing 1.5 lakhs",
]

print("Extracting from 3 different texts — same schema every time:")
print("=" * 60)

for text in texts:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract product information from the user's message."},
            {"role": "user", "content": text}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "product_extraction",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "product_name": {"type": "string"},
                        "price_per_unit": {"type": "number"},
                        "quantity": {"type": "integer"},
                        "total_amount": {"type": "number"}
                    },
                    "required": ["product_name", "price_per_unit", "quantity", "total_amount"],
                    "additionalProperties": False
                }
            }
        }
    )
    result = json.loads(response.choices[0].message.content)
    print(f"\n📦 \"{text}\"")
    print(f"   → {result}")

### Method 1 vs Method 2 — When to Use Which

| | `json_object` | `json_schema` |
|---|---|---|
| **Guarantees** | Valid JSON | Valid JSON + exact keys + exact types |
| **Control** | Low (LLM picks keys) | Full (you define everything) |
| **Use when** | Quick prototyping | Production pipelines |
| **Analogy** | "Write me a letter" | "Fill this form" |

> ⚠️ **Common Mistakes**
> 1. Using `json_object` without telling the LLM what keys you want (in system prompt) — it will pick random keys
> 2. Forgetting `"additionalProperties": false` — LLM may add extra fields you didn't ask for
> 3. Not setting `"strict": true` — without this, the schema is just a suggestion, not enforced

---
## Part 3: Output Validation — Trust But Verify

Even with structured outputs, things can go wrong:
- The LLM might put `"price": "twenty five thousand"` instead of a number
- A field might be `null` when you expected a value
- The JSON might be syntactically valid but logically wrong

**Validation = checking that the response makes sense BEFORE your code uses it.**

```
LLM Response  →  Parse JSON  →  Validate fields  →  Use in your app
                      │               │
                      ▼               ▼
                  If invalid:      If missing/wrong:
                  retry or fail    retry or use default
```

In [10]:
# ──────────────────────────────────────
# A simple validator function
# ──────────────────────────────────────

def validate_product_response(raw_response):
    """
    Validate that the LLM's response has the correct shape.
    Returns (is_valid, data_or_error)
    """
    # Step 1: Is it valid JSON?
    try:
        data = json.loads(raw_response)
    except json.JSONDecodeError as e:
        return False, f"Not valid JSON: {e}"
    
    # Step 2: Does it have the required keys?
    required_keys = ["product_name", "price_per_unit", "quantity", "total_amount"]
    missing = [k for k in required_keys if k not in data]
    if missing:
        return False, f"Missing keys: {missing}"
    
    # Step 3: Are the types correct?
    if not isinstance(data["product_name"], str):
        return False, "product_name must be a string"
    if not isinstance(data["price_per_unit"], (int, float)):
        return False, "price_per_unit must be a number"
    if not isinstance(data["quantity"], int):
        return False, "quantity must be an integer"
    
    # Step 4: Do the values make sense?
    if data["price_per_unit"] <= 0:
        return False, "price_per_unit must be positive"
    if data["quantity"] <= 0:
        return False, "quantity must be positive"
    
    return True, data

# Test with good data
good = '{"product_name": "Samsung Galaxy", "price_per_unit": 25000, "quantity": 3, "total_amount": 75000}'
is_valid, result = validate_product_response(good)
print(f"✅ Good data: valid={is_valid}, result={result}")

# Test with bad data
bad_json = "This is not JSON at all"
is_valid, result = validate_product_response(bad_json)
print(f"❌ Bad JSON:  valid={is_valid}, error={result}")

# Test with missing key
missing = '{"product_name": "Phone", "price_per_unit": 25000}'
is_valid, result = validate_product_response(missing)
print(f"❌ Missing:   valid={is_valid}, error={result}")

# Test with bad value
bad_value = '{"product_name": "Phone", "price_per_unit": -500, "quantity": 3, "total_amount": -1500}'
is_valid, result = validate_product_response(bad_value)
print(f"❌ Bad value: valid={is_valid}, error={result}")

✅ Good data: valid=True, result={'product_name': 'Samsung Galaxy', 'price_per_unit': 25000, 'quantity': 3, 'total_amount': 75000}
❌ Bad JSON:  valid=False, error=Not valid JSON: Expecting value: line 1 column 1 (char 0)
❌ Missing:   valid=False, error=Missing keys: ['quantity', 'total_amount']
❌ Bad value: valid=False, error=price_per_unit must be positive


In [ ]:
# ──────────────────────────────────────
# Combining: Structured Output + Validation + Retry
# ──────────────────────────────────────

def extract_product_info(text, max_retries=3):
    """
    Extract product info from text with:
    1. Structured output (forces JSON)
    2. Validation (checks correctness)
    3. Retry (tries again on failure)
    """
    for attempt in range(max_retries):
        # Call LLM with structured output
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Extract product information from the text."},
                {"role": "user", "content": text}
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "product_extraction",
                    "strict": True,
                    "schema": {
                        "type": "object",
                        "properties": {
                            "product_name": {"type": "string"},
                            "price_per_unit": {"type": "number"},
                            "quantity": {"type": "integer"},
                            "total_amount": {"type": "number"}
                        },
                        "required": ["product_name", "price_per_unit", "quantity", "total_amount"],
                        "additionalProperties": False
                    }
                }
            }
        )
        
        raw = response.choices[0].message.content
        
        # Validate
        is_valid, result = validate_product_response(raw)
        
        if is_valid:
            return result  # ✅ Success
        else:
            print(f"  ⚠️ Attempt {attempt + 1} failed validation: {result}")
    
    return None  # All retries failed

# Test
info = extract_product_info("I need 5 HP printers at 15000 each")
print("Extracted:")
print(json.dumps(info, indent=2))

> 💡 **Key Pattern**: `Call → Parse → Validate → Use (or Retry)`
>
> This is the pattern for every reliable LLM pipeline. You never trust the output blindly.

---
## Part 4: Structured Outputs in Tool Responses

In Session 04, our tools returned free-form strings:

```python
# Session 04 style (free text)
def get_weather(city):
    return "32°C, Partly Cloudy"  # ← unstructured
```

In production, tools should return **structured JSON** so the LLM (and your code) can reliably parse them:

```python
# Session 05 style (structured)
def get_weather(city):
    return json.dumps({"city": city, "temp_c": 32, "condition": "Partly Cloudy"})  # ← structured
```

### Why This Matters

When your tool returns structured data:
1. The LLM can extract specific values reliably
2. Your code can process the tool result without guessing
3. Errors are clearly separated from valid results

In [11]:
# ──────────────────────────────────────
# Upgraded tools with structured responses
# ──────────────────────────────────────

# Employee data (same as Session 04)
import csv
from io import StringIO

CSV_DATA = """name,department,salary,experience_years,city
Rahul,Engineering,75000,5,Hyderabad
Priya,Marketing,55000,3,Bangalore
Arjun,Engineering,82000,7,Hyderabad
Sneha,HR,48000,2,Mumbai
Vikram,Engineering,90000,9,Delhi
Anita,Marketing,60000,4,Bangalore
Karthik,HR,52000,3,Chennai
Deepa,Engineering,70000,4,Hyderabad
Suresh,Marketing,65000,6,Mumbai
Kavya,HR,50000,2,Bangalore
Ravi,Engineering,95000,10,Delhi
Meera,Marketing,58000,3,Chennai
Arun,Engineering,78000,6,Hyderabad
Lavanya,HR,55000,4,Bangalore
Sanjay,Marketing,62000,5,Mumbai
"""

reader = csv.DictReader(StringIO(CSV_DATA.strip()))
employees = list(reader)

# ─── Better tools with structured output ───

def get_stats(column: str) -> str:
    """Get stats for a column. Returns structured JSON."""
    try:
        values = [float(row[column]) for row in employees]
        return json.dumps({
            "status": "success",
            "column": column,
            "count": len(values),
            "mean": round(sum(values) / len(values), 2),
            "min": min(values),
            "max": max(values)
        })
    except (ValueError, KeyError):
        return json.dumps({
            "status": "error",
            "message": f"Column '{column}' is not numeric or doesn't exist",
            "available_numeric_columns": ["salary", "experience_years"]
        })

def filter_data(column: str, value: str) -> str:
    """Filter rows and return structured results."""
    try:
        matches = [row for row in employees if row[column].lower() == value.lower()]
        return json.dumps({
            "status": "success",
            "filter": {"column": column, "value": value},
            "count": len(matches),
            "results": matches
        })
    except KeyError:
        return json.dumps({
            "status": "error",
            "message": f"Column '{column}' not found",
            "available_columns": list(employees[0].keys())
        })

# Compare: old style vs new style
print("── Structured tool response: ──")
print(get_stats("salary"))
print()
print("── Error response (also structured): ──")
print(get_stats("age"))  # column doesn't exist

── Structured tool response: ──
{"status": "success", "column": "salary", "count": 15, "mean": 66333.33, "min": 48000.0, "max": 95000.0}

── Error response (also structured): ──
{"status": "error", "message": "Column 'age' is not numeric or doesn't exist", "available_numeric_columns": ["salary", "experience_years"]}


Notice how both success AND error follow a pattern:

```json
// Success
{"status": "success", "column": "salary", "mean": 66333, ...}

// Error
{"status": "error", "message": "...", "available_columns": [...]}
```

This means the LLM (and your code) can always check `status` first to know if it worked.

---
## Part 5: Idempotent Tool Design

### What Does "Idempotent" Mean?

A tool is **idempotent** if calling it twice produces the same result and doesn't cause harm.

```
IDEMPOTENT (safe to run twice):
  ✅ get_weather("Hyderabad")  → 32°C (same result every time)
  ✅ get_stats("salary")       → mean=66333 (same result every time)
  ✅ read_file("data.csv")     → returns file contents (doesn't change anything)

NOT IDEMPOTENT (dangerous to run twice):
  ❌ send_email(to="boss", body="...")  → sends ANOTHER email if called again!
  ❌ transfer_money(amount=5000)         → transfers AGAIN if called again!
  ❌ delete_record(id=42)                → errors on second call (already deleted)
```

### Why Does This Matter for Agents?

Remember: agents have a **retry loop**. If something fails midway, the agent might call the same tool again. If that tool sends an email or transfers money — you've got a problem.

```
Agent: "Send confirmation email"  → Email sent ✅
Agent: *network error, retries*
Agent: "Send confirmation email"  → ANOTHER email sent 😱
```

In [ ]:
# ──────────────────────────────────────
# Example: Non-idempotent vs Idempotent tool
# ──────────────────────────────────────

# ❌ BAD: Not idempotent — creates duplicate orders
orders_bad = []

def place_order_bad(product: str, quantity: int) -> str:
    """Place an order (NOT idempotent — dangerous!)."""
    order = {"product": product, "quantity": quantity, "id": len(orders_bad) + 1}
    orders_bad.append(order)
    return json.dumps({"status": "success", "order": order})

# Simulate agent calling it twice (due to retry)
print("❌ NON-IDEMPOTENT tool (called twice):")
print(place_order_bad("Laptop", 1))
print(place_order_bad("Laptop", 1))  # Duplicate order!
print(f"   Orders in system: {len(orders_bad)} ← Should be 1, but it's 2! 😱")
print()

In [ ]:
# ✅ GOOD: Idempotent — uses a unique request_id to prevent duplicates
orders_good = {}

def place_order_safe(product: str, quantity: int, request_id: str) -> str:
    """Place an order (idempotent — safe to retry)."""
    # If this request_id was already processed, return the same result
    if request_id in orders_good:
        return json.dumps({
            "status": "success",
            "message": "Already processed (duplicate request)",
            "order": orders_good[request_id]
        })
    
    # First time — create the order
    order = {"product": product, "quantity": quantity, "request_id": request_id}
    orders_good[request_id] = order
    return json.dumps({"status": "success", "order": order})

# Simulate agent calling it twice with same request_id
print("✅ IDEMPOTENT tool (called twice with same request_id):")
print(place_order_safe("Laptop", 1, request_id="order_abc123"))
print(place_order_safe("Laptop", 1, request_id="order_abc123"))  # Same ID = no duplicate
print(f"   Orders in system: {len(orders_good)} ← Correctly just 1! ✅")

### Idempotency Rules of Thumb

| Tool Type | Idempotent? | How to Make Safe |
|---|---|---|
| **Read** data (get, list, search) | ✅ Already safe | No change needed |
| **Calculate** something | ✅ Already safe | No change needed |
| **Create** something (order, email) | ❌ Dangerous | Use a `request_id` to detect duplicates |
| **Update** a record | ⚠️ Depends | Set to final value, don't increment |
| **Delete** a record | ⚠️ Depends | Check if already deleted before deleting |

> 💡 **Simple Rule**: If your tool only READS data, it's already idempotent. If it CHANGES data, add a `request_id` or check before acting.

---
## Part 6: Putting It All Together — A Reliable Agent Pipeline

Let's combine everything into one clean agent that:
1. Uses **structured output** for the final answer
2. Has **validated tool responses**
3. Uses **structured tools** (JSON in, JSON out)
4. Has **retry + hard stop**

We'll build an **Employee Report Agent** — you ask a question, it returns a structured report.

In [ ]:
# ──────────────────────────────────────
# Tool definitions (JSON schemas)
# ──────────────────────────────────────

report_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_stats",
            "description": "Get statistics (mean, min, max, count) for a numeric column. Available columns: salary, experience_years.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {"type": "string", "description": "Numeric column name"}
                },
                "required": ["column"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "filter_data",
            "description": "Filter employees by a column value. Available columns: name, department, city. Returns matching employees.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {"type": "string", "description": "Column to filter on"},
                    "value": {"type": "string", "description": "Value to match"}
                },
                "required": ["column", "value"]
            }
        }
    }
]

report_available_tools = {
    "get_stats": get_stats,
    "filter_data": filter_data,
}

print("✅ Report tools ready")

In [ ]:
# ──────────────────────────────────────
# The Reliable Report Agent
# ──────────────────────────────────────

def generate_report(question, verbose=True):
    """
    Ask a question → Agent uses tools → Returns a STRUCTURED report.
    
    The final output is always:
    {
        "question": "...",
        "answer": "...",
        "data_used": [...],
        "confidence": "high" | "medium" | "low"
    }
    """
    
    # Phase 1: Let agent gather data using tools
    messages = [
        {"role": "system", "content": (
            "You are a data analyst. Use tools to gather data, then answer the question. "
            "Use tools first before answering."
        )},
        {"role": "user", "content": question}
    ]
    
    if verbose:
        print(f"\n{'═' * 60}")
        print(f"📊 Question: {question}")
        print(f"{'─' * 60}")
    
    # Agent loop (gather data)
    max_steps = 5
    tool_results = []
    
    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=report_tools,
        )
        
        choice = response.choices[0]
        
        if choice.finish_reason == "stop":
            break
        
        if choice.message.tool_calls:
            messages.append(choice.message)
            for tool_call in choice.message.tool_calls:
                func_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)
                
                if verbose:
                    print(f"  🔧 {func_name}({args})")
                
                result = report_available_tools[func_name](**args)
                tool_results.append({"tool": func_name, "args": args, "result": result})
                
                if verbose:
                    print(f"     → {result[:80]}..." if len(result) > 80 else f"     → {result}")
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
    
    # Phase 2: Generate structured report from gathered data
    if verbose:
        print(f"{'─' * 60}")
        print("  📝 Generating structured report...")
    
    report_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Generate a structured report based on the data analysis."},
            {"role": "user", "content": f"Question: {question}\n\nData gathered: {json.dumps(tool_results)}\n\nGenerate a report."}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "analysis_report",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string"},
                        "answer": {"type": "string"},
                        "key_numbers": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "confidence": {
                            "type": "string",
                            "description": "high, medium, or low"
                        }
                    },
                    "required": ["question", "answer", "key_numbers", "confidence"],
                    "additionalProperties": False
                }
            }
        }
    )
    
    
    report = json.loads(report_response.choices[0].message.content)
    return report

print("✅ Report agent ready!")

In [ ]:
# Test the report agent
report = generate_report("What's the salary situation in the Engineering department?")

print("\n📋 STRUCTURED REPORT:")
print(json.dumps(report, indent=2))

In [ ]:
# Another question
report = generate_report("Compare the average experience between departments")

print("\n📋 STRUCTURED REPORT:")
print(json.dumps(report, indent=2))

# Access specific fields programmatically
print(f"\n── Programmatic access (no parsing needed): ──")
print(f"Answer: {report['answer']}")
print(f"Confidence: {report['confidence']}")
print(f"Key numbers: {report['key_numbers']}")

In [ ]:
# One more — showing multi-step tool usage
report = generate_report("How many people are in Hyderabad and what's their average salary?")

print("\n📋 STRUCTURED REPORT:")
print(json.dumps(report, indent=2))

---
## Part 7: The Full Picture — Before vs After

```
┌────────────────────────────────────────────────────────────┐
│  SESSION 04 (Before)                                       │
│                                                            │
│  User → Agent → Tools → Free text answer                   │
│                                                            │
│  Problems:                                                 │
│  • Output format changes every time                        │
│  • Can't reliably extract numbers from response            │
│  • Tool errors mixed with valid results                    │
│  • Re-running a tool might cause duplicates                │
└────────────────────────────────────────────────────────────┘

                           ▼

┌────────────────────────────────────────────────────────────┐
│  SESSION 05 (After)                                        │
│                                                            │
│  User → Agent → Structured Tools → Validated → Structured  │
│                                      Output     Report     │
│                                                            │
│  Improvements:                                             │
│  ✅ Output is ALWAYS the same JSON shape                   │
│  ✅ Tools return {status, data} — errors are clear         │
│  ✅ Validation catches bad responses before they break code│
│  ✅ Idempotent tools are safe to retry                     │
└────────────────────────────────────────────────────────────┘
```

---
## Summary — Key Concepts

| Concept | One-liner | Code |
|---|---|---|
| **Structured Output** | Force LLM to output exact JSON shape | `response_format={"type": "json_schema", ...}` |
| **JSON Mode (simple)** | Force valid JSON (any shape) | `response_format={"type": "json_object"}` |
| **JSON Schema (strict)** | Force exact keys, types, structure | `"strict": True` + full schema |
| **Validation** | Check response before using it | Parse → check keys → check types → check values |
| **Idempotent Tools** | Safe to call twice | Use `request_id` for write operations |
| **Structured Tool Response** | Tools return JSON, not free text | `return json.dumps({"status": ..., "data": ...})` |

### The Reliable Pipeline Pattern

```python
# Every production LLM pipeline follows this:

response = call_llm(structured_output=True)  # 1. Force structure
data = json.loads(response)                   # 2. Parse
is_valid = validate(data)                     # 3. Validate
if not is_valid:                              # 4. Retry or fail
    retry()
use(data)                                     # 5. Use confidently
```

### What's Next?

- **RAG (Retrieval-Augmented Generation)** — Give the agent access to your own documents
- **Multi-step pipelines** — Chain multiple agents together
- **Evaluation** — How to measure if your agent is actually good